# Feature Engineering and Data Preprocessing

The objective of this notebook is to transform the raw Home Credit Default Risk dataset into a machine-learning-ready dataset.

This includes:

- Feature selection
- Missing value treatment
- Missing indicator creation
- Placeholder value correction
- Feature engineering
- Categorical encoding
- Dataset export

The resulting dataset will be used for model development in the next phase of the project.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Libraries loaded successfully.")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

In [ ]:
train_df = pd.read_csv("../data/raw/application_train.csv")

print(train_df.shape)

In [ ]:
train_df.head()

## Initial Dataset Shape

The raw dataset is loaded from the Home Credit Default Risk application training data.

All preprocessing operations will be applied to this dataset before model training.

In [ ]:
drop_columns = [
    "YEARS_BUILD_AVG",
    "YEARS_BUILD_MEDI",
    "YEARS_BUILD_MODE",

    "COMMONAREA_AVG",
    "COMMONAREA_MEDI",
    "COMMONAREA_MODE",

    "FLOORSMIN_AVG",
    "FLOORSMIN_MEDI",
    "FLOORSMIN_MODE",

    "LIVINGAPARTMENTS_AVG",
    "LIVINGAPARTMENTS_MEDI",
    "LIVINGAPARTMENTS_MODE",

    "NONLIVINGAPARTMENTS_AVG",
    "NONLIVINGAPARTMENTS_MEDI",
    "NONLIVINGAPARTMENTS_MODE",

    "FONDKAPREMONT_MODE",
    "HOUSETYPE_MODE",
    "WALLSMATERIAL_MODE",
    "EMERGENCYSTATE_MODE"
]

print("Columns scheduled for removal:", len(drop_columns))

In [ ]:
train_df = train_df.drop(columns=drop_columns)

print(train_df.shape)

## High-Missing Feature Removal

Features exhibiting extremely high levels of missingness and limited business value were removed.

This step reduces noise and simplifies subsequent preprocessing operations while retaining the most informative variables.

## Missing Value Indicator Features

Certain missing values may themselves contain predictive information.

Indicator variables are created to preserve information about missingness while allowing numerical imputation to be performed later.

These engineered features enable the model to distinguish between genuinely observed values and values that were originally missing.

In [ ]:
train_df["EXT_SOURCE_1_MISSING"] = (
    train_df["EXT_SOURCE_1"]
    .isnull()
    .astype(int)
)

train_df["EXT_SOURCE_1_MISSING"].value_counts()

In [ ]:
train_df["OWN_CAR_AGE_MISSING"] = (
    train_df["OWN_CAR_AGE"]
    .isnull()
    .astype(int)
)

train_df["OWN_CAR_AGE_MISSING"].value_counts()

In [ ]:
[
    col
    for col in train_df.columns
    if "MISSING" in col
]

### Observations

The missing-value indicators confirm that EXT_SOURCE_1 and OWN_CAR_AGE contain substantial levels of missing data.

Previous exploratory analysis demonstrated that both features possess predictive value, and missingness itself may contain meaningful information.

Indicator variables have therefore been created to preserve this information while enabling numerical imputation in later preprocessing stages.

These engineered features may improve model performance by allowing the algorithm to distinguish between observed and imputed values.

## Employment Data Quality Correction

The DAYS_EMPLOYED feature contains placeholder values equal to 365243 days.

These values do not represent actual employment durations and must be treated as missing values.

An indicator feature will be created to preserve the information contained in these placeholder records before correction.

In [ ]:
(train_df["DAYS_EMPLOYED"] == 365243).sum()

In [ ]:
train_df["DAYS_EMPLOYED_PLACEHOLDER"] = (
    train_df["DAYS_EMPLOYED"] == 365243
).astype(int)

train_df["DAYS_EMPLOYED_PLACEHOLDER"].value_counts()

In [ ]:
train_df["DAYS_EMPLOYED"] = train_df["DAYS_EMPLOYED"].replace(
    365243,
    np.nan
)

train_df["DAYS_EMPLOYED"].isnull().sum()

In [ ]:
(train_df["DAYS_EMPLOYED"] == 365243).sum()

## Feature Engineering

Additional features are created to improve the representation of applicant characteristics and creditworthiness.

These engineered features are designed to enhance model performance by capturing more meaningful business information than the original variables alone.

The transformations focus on applicant age, employment history, and external credit score information.

In [ ]:
train_df["AGE_YEARS"] = abs(train_df["DAYS_BIRTH"]) / 365

train_df["AGE_YEARS"].describe()

In [ ]:
train_df["EMPLOYMENT_YEARS"] = (
    abs(train_df["DAYS_EMPLOYED"]) / 365
)

train_df["EMPLOYMENT_YEARS"].describe()

In [ ]:
train_df["EXT_SOURCE_MEAN"] = train_df[
    [
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3"
    ]
].mean(axis=1)

train_df["EXT_SOURCE_MEAN"].describe()

### Engineered Feature Validation

The engineered features exhibit realistic distributions and expected value ranges.

AGE_YEARS provides an interpretable representation of applicant age.

EMPLOYMENT_YEARS reflects employment duration after correcting placeholder values.

EXT_SOURCE_MEAN successfully aggregates multiple external credit indicators while substantially reducing missingness compared to individual source variables.

In [ ]:
engineered_features = [
    "AGE_YEARS",
    "EMPLOYMENT_YEARS",
    "EXT_SOURCE_MEAN"
]

train_df[engineered_features].head()

In [ ]:
train_df[
    [
        "TARGET",
        "AGE_YEARS",
        "EMPLOYMENT_YEARS",
        "EXT_SOURCE_MEAN"
    ]
].corr()["TARGET"].sort_values()

### Correlation Analysis of Engineered Features

The engineered features demonstrate meaningful relationships with loan default risk.

EXT_SOURCE_MEAN exhibits the strongest predictive signal among the engineered variables, indicating that aggregated external credit information is highly informative for credit risk assessment.

AGE_YEARS and EMPLOYMENT_YEARS both show negative correlations with default risk, suggesting that older applicants and individuals with longer employment histories tend to exhibit more reliable repayment behavior.

These findings validate the effectiveness of the feature engineering process and support the inclusion of these variables in the final modeling dataset.

## Missing Value Imputation Strategy

Machine learning algorithms generally require complete datasets without missing values.

Based on the preprocessing strategy developed during exploratory analysis, numerical variables will be imputed using median values, while categorical variables will be imputed using appropriate category-based methods.

This approach preserves the integrity of the dataset while minimizing the influence of extreme values.

In [ ]:
missing_after_engineering = (
    train_df.isnull().sum() / len(train_df)
) * 100

missing_after_engineering = (
    missing_after_engineering[missing_after_engineering > 0]
    .sort_values(ascending=False)
)

missing_after_engineering.head(20)

In [ ]:
missing_after_engineering.count()

In [ ]:
missing_after_engineering.tail(20)

In [ ]:
print(missing_after_engineering)

## Missing Value Assessment

A detailed review of missing values was performed after feature engineering and data cleaning.

A total of 51 features still contained missing values, with missingness ranging from less than 1% to approximately 66%. High-missingness features such as OWN_CAR_AGE and EXT_SOURCE_1 were retained because earlier exploratory analysis demonstrated predictive value, and additional missing-value indicator features were created where appropriate.

Rather than removing a large number of potentially informative variables, a structured imputation strategy will be applied. This approach preserves valuable business information while ensuring compatibility with machine learning algorithms that require complete datasets.

The next step focuses on imputing numerical variables using median values and handling categorical missing values using category-based techniques.

## Numerical Missing Value Imputation

Numerical variables containing missing values are imputed using median values.

Median imputation is robust to outliers and preserves the overall distribution of skewed financial variables.

This approach is particularly suitable for credit-risk datasets where extreme values are common.

In [ ]:
numeric_missing_cols = train_df.select_dtypes(
    include=["int64", "float64"]
).columns

numeric_missing_cols = [
    col
    for col in numeric_missing_cols
    if train_df[col].isnull().sum() > 0
]

print("Numeric columns with missing values:")
print(len(numeric_missing_cols))

In [ ]:
for col in numeric_missing_cols:
    train_df[col] = train_df[col].fillna(
        train_df[col].median()
    )

print("Numeric imputation completed.")

In [ ]:
train_df[numeric_missing_cols].isnull().sum().sum()

### Observations

Numerical missing value treatment was successfully completed.

A total of 49 numerical features contained missing values and were imputed using median values. Median imputation was selected because financial and credit-related variables often contain skewed distributions and extreme values, making the median a more robust measure than the mean.

After imputation, all numerical features contain valid values and no numerical missing data remains in the dataset. This ensures compatibility with subsequent machine learning algorithms while preserving the overall distribution of the original variables.

## Categorical Missing Value Treatment

The remaining missing values are primarily located in categorical features.

These variables require category-based imputation strategies to preserve business meaning while ensuring that the dataset is fully complete for model training.

In [ ]:
remaining_missing = (
    train_df.isnull().sum()
)

remaining_missing = (
    remaining_missing[remaining_missing > 0]
    .sort_values(ascending=False)
)

remaining_missing

### Categorical Imputation Strategy

Different imputation methods are applied based on the level of missingness and business context.

OCCUPATION_TYPE contains a substantial proportion of missing values and will be assigned a dedicated "Unknown" category to preserve potential information contained within the missingness itself.

NAME_TYPE_SUITE contains very few missing values and will therefore be imputed using the most frequently occurring category.

In [ ]:
train_df["OCCUPATION_TYPE"] = (
    train_df["OCCUPATION_TYPE"]
    .fillna("Unknown")
)

train_df["OCCUPATION_TYPE"].isnull().sum()

In [ ]:
suite_mode = (
    train_df["NAME_TYPE_SUITE"]
    .mode()[0]
)

train_df["NAME_TYPE_SUITE"] = (
    train_df["NAME_TYPE_SUITE"]
    .fillna(suite_mode)
)

train_df["NAME_TYPE_SUITE"].isnull().sum()

In [ ]:
train_df.isnull().sum().sum()

### Observations

Categorical missing values were successfully handled using feature-specific imputation strategies.

OCCUPATION_TYPE was assigned an "Unknown" category due to its high proportion of missing values, while NAME_TYPE_SUITE was imputed using its most frequent category.

Following both numerical and categorical imputation, the dataset contains no remaining missing values. The data is now complete and ready for categorical encoding and machine learning model development.

In [ ]:
train_df.shape

## Preprocessing Completion Summary

The dataset has undergone feature selection, missing value analysis, feature engineering, and imputation.

High-missing-value features were removed, business-oriented features were engineered, and both numerical and categorical missing values were treated using appropriate imputation strategies.

The resulting dataset contains 307,511 records and 109 features with no remaining missing values, making it suitable for categorical encoding and machine learning model development.

## Categorical Encoding Strategy

Categorical variables must be converted into numerical representations before model training.

Low-cardinality categorical variables will be transformed using one-hot encoding to preserve category-specific information.

High-cardinality variables such as ORGANIZATION_TYPE and OCCUPATION_TYPE will be transformed using frequency encoding to avoid excessive dimensionality while retaining useful distributional information.

In [ ]:
categorical_cols = train_df.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("Number of categorical columns:", len(categorical_cols))
print(categorical_cols)

### Encoding Preparation

A review of categorical variables identified 12 remaining categorical features.

Low-cardinality variables will be transformed using one-hot encoding, while higher-cardinality variables such as OCCUPATION_TYPE and ORGANIZATION_TYPE will be frequency encoded to reduce dimensionality and improve model efficiency.

In [ ]:
for col in ["OCCUPATION_TYPE", "ORGANIZATION_TYPE"]:
    
    freq_map = train_df[col].value_counts(normalize=True)
    
    train_df[col + "_FREQ"] = train_df[col].map(freq_map)

train_df[
    [
        "OCCUPATION_TYPE_FREQ",
        "ORGANIZATION_TYPE_FREQ"
    ]
].head()

In [ ]:
occupation_frequency = (
    train_df["OCCUPATION_TYPE"]
    .value_counts(normalize=True)
    .to_dict()
)

organization_frequency = (
    train_df["ORGANIZATION_TYPE"]
    .value_counts(normalize=True)
    .to_dict()
)

print("Occupation categories:", len(occupation_frequency))
print("Organization categories:", len(organization_frequency))

In [ ]:
import pickle

with open("../models/occupation_frequency.pkl", "wb") as f:
    pickle.dump(occupation_frequency, f)

with open("../models/organization_frequency.pkl", "wb") as f:
    pickle.dump(organization_frequency, f)

print("Frequency dictionaries saved.")

In [ ]:
train_df = train_df.drop(
    columns=[
        "OCCUPATION_TYPE",
        "ORGANIZATION_TYPE"
    ]
)

train_df.shape

In [ ]:
one_hot_cols = [
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "NAME_TYPE_SUITE",
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "WEEKDAY_APPR_PROCESS_START"
]

train_df = pd.get_dummies(
    train_df,
    columns=one_hot_cols,
    drop_first=True
)

train_df.shape

In [ ]:
print("Shape:", train_df.shape)

print(
    "Remaining categorical columns:",
    len(
        train_df.select_dtypes(
            include=["object", "string"]
        ).columns
    )
)

In [ ]:
train_df.isnull().sum().sum()

## Encoding Summary

Categorical variables were successfully transformed into machine-learning-compatible numerical representations.

High-cardinality features (OCCUPATION_TYPE and ORGANIZATION_TYPE) were frequency encoded to preserve information while avoiding excessive dimensionality.

The remaining low-cardinality categorical variables were transformed using one-hot encoding with drop_first=True to reduce redundancy and multicollinearity.

Following encoding, the dataset contains 307,511 observations and 137 numerical features with no remaining missing values.

In [ ]:
train_df.info()

In [ ]:
bool_cols = train_df.select_dtypes(include=["bool"]).columns

train_df[bool_cols] = train_df[bool_cols].astype(int)

train_df.info()

### Boolean Feature Conversion

One-hot encoded features were initially stored as boolean values.

These variables were converted to integer representations (0 and 1) to maintain a fully numerical dataset and ensure compatibility across machine learning libraries and deployment environments.

In [ ]:
print("Shape:", train_df.shape)

print(
    "Total Missing Values:",
    train_df.isnull().sum().sum()
)

print(
    "Categorical Columns:",
    len(
        train_df.select_dtypes(
            include=["object", "string"]
        ).columns
    )
)

### LightGBM Feature Name Compatibility

LightGBM does not support certain special characters in feature names. After one-hot encoding, some generated column names contained characters such as "/", ",", ":", " ".

To ensure compatibility, feature names were standardized by replacing non-alphanumeric characters with underscores. This transformation affects only column names and does not modify the underlying data or model features.

In [ ]:
train_df.columns = (
    train_df.columns
    .str.replace("/", "_", regex=False)
    .str.replace(",", "_", regex=False)
    .str.replace(" ", "_", regex=False)
    .str.replace(":", "_", regex=False)
)

In [ ]:
problem_cols = [
    col for col in train_df.columns
    if any(char in col for char in ["/", ",", " ",":"])
]

problem_cols

In [ ]:
train_df.to_csv(
    "../data/processed/train_processed.csv",
    index=False
)

print("Processed dataset saved successfully.")

In [ ]:
import os

os.path.exists(
    "../data/processed/train_processed.csv"
)

# Preprocessing Pipeline Completed

The dataset has successfully passed through the preprocessing pipeline.

Key preprocessing tasks completed:

- Feature selection
- Missing value analysis
- Missing value indicators
- Placeholder value correction
- Feature engineering
- Numerical imputation
- Categorical imputation
- Frequency encoding
- One-hot encoding
- Dataset validation

Final Dataset Summary:

- Records: 307,511
- Features: 137
- Missing Values: 0
- Categorical Columns: 0

The processed dataset is now ready for machine learning model development and evaluation.